In [44]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [45]:
X, y = make_classification(n_samples=10000, n_features=10, n_informative=3)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [46]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)

print("Decision Tree Accuracy", accuracy_score(y_test, y_pred))

Decision Tree Accuracy 0.921


### Bagging using Decision Tree

In [47]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)

In [48]:
bag.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, random_state=42)

In [49]:
y_pred = bag.predict(X_test)

In [50]:
accuracy_score(y_test, y_pred)

0.9485

In [51]:
bag.estimators_samples_[0].shape

(2000,)

In [52]:
bag.estimators_features_[0].shape

(10,)

### Bagging using SVM

In [53]:
bag = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)

In [54]:
bag.fit(X_train, y_train)

y_pred = bag.predict(X_test)

print("Bagging using SVM", accuracy_score(y_test, y_pred))

Bagging using SVM 0.939


### Pasting

In [55]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

In [56]:
bag.fit(X_train, y_train)

y_pred = bag.predict(X_test)

print("Pasting Classifier", accuracy_score(y_test, y_pred))

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:   11.9s remaining:  1.0min
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:   12.1s finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


Pasting Classifier 0.951


[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:    0.0s remaining:    0.3s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:    0.1s finished


### Random Subspaces

In [57]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    max_features=0.5,
    bootstrap_features=True,
    bootstrap=False,
    random_state=42
)

In [58]:
bag.fit(X_train, y_train)

y_pred = bag.predict(X_test)

print("Random Subspace Classifier", accuracy_score(y_test, y_pred))

Random Subspace Classifier 0.9495


In [59]:
bag.estimators_samples_[0].shape

(8000,)

In [60]:
bag.estimators_features_[0].shape

(5,)

### Random Patches

In [61]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    max_features=0.5,
    bootstrap_features=True,
    bootstrap=True,
    random_state=42
)

In [62]:
bag.fit(X_train, y_train)

y_pred = bag.predict(X_test)

print("Random Patch Classifier", accuracy_score(y_test, y_pred))

Random Patch Classifier 0.947


### OOB Score

In [63]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

In [64]:
bag.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, oob_score=True, random_state=42)

In [65]:
bag.oob_score_

0.94575

In [66]:
y_pred = bag.predict(X_test)

print("OOB Accuracy", accuracy_score(y_test, y_pred))

OOB Accuracy 0.9485


### Bagging Tips
- Bagging generally gives better results than Pasting
- Good results come around the 25% to 50% row sampling mark
- Random Patches and Subspaces should be used while dealing with high dimensional data
- To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

### Applying GridSearchCV

In [67]:
from sklearn.model_selection import GridSearchCV

In [68]:
parameters = {
    'n_estimators': [50, 100, 500],
    'max_samples': [0.1, 0.4, 0.7, 1.0],
    'bootstrap': [True, False],
    'max_features': [0.1, 0.4, 0.7, 1.0] 
}

In [69]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5)

In [70]:
search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=BaggingClassifier(),
             param_grid={'bootstrap': [True, False],
                         'max_features': [0.1, 0.4, 0.7, 1.0],
                         'max_samples': [0.1, 0.4, 0.7, 1.0],
                         'n_estimators': [50, 100, 500]})

In [71]:
search.best_params_

{'bootstrap': True,
 'max_features': 0.7,
 'max_samples': 1.0,
 'n_estimators': 100}

In [72]:
search.best_score_

0.9504999999999999